In [2]:
import os
import json
import numpy as np
import pandas as pd
import pickle
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from collections import defaultdict
from transformers import CamembertTokenizer, CamembertModel

## A. Data Pre-processing

1️⃣ Load JSONL → Extract tweet_id, user_id, full_text

In [3]:
import json
import pandas as pd

TEXT_COL = "clean_text"
TWEET_ID_COL = "tweet_id"
USER_ID_COL = "user_id"

def load_jsonl(path):
    # In your load_jsonl function or right after opening the file
    with open("data/train.jsonl", "r", encoding="utf-8") as f:
        first_line = f.readline()
        sample_tweet = json.loads(first_line)
        
        print("Top-level keys in tweet:")
        print(sample_tweet.keys())
        
        print("\nKeys in user dict:")
        print(sample_tweet.get("user", {}).keys())
        
        print("\nLooking for challenge_id:")
        print("  At tweet level:", sample_tweet.get("challenge_id"))
        print("  In user dict:", sample_tweet.get("user", {}).get("challenge_id"))
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            d = json.loads(line)
            
            tweet_id = d.get("id_str")
            user_id = d.get("user", {}).get("challenge_id")
            
            # extract full tweet text (priority: extended_tweet.full_text)
            if "extended_tweet" in d and "full_text" in d["extended_tweet"]:
                text = d["extended_tweet"]["full_text"]
            else:
                text = d.get("text", "")
            
            rows.append({
                TWEET_ID_COL: tweet_id,
                USER_ID_COL: user_id,
                TEXT_COL: text
            })
    return pd.DataFrame(rows)

train_df = load_jsonl("data/train.jsonl")
kaggle_df = load_jsonl("data/kaggle_test.jsonl")


Top-level keys in tweet:
dict_keys(['quoted_status', 'in_reply_to_status_id_str', 'in_reply_to_status_id', 'created_at', 'in_reply_to_user_id_str', 'source', 'quoted_status_id', 'retweet_count', 'retweeted', 'geo', 'filter_level', 'in_reply_to_screen_name', 'is_quote_status', 'id_str', 'in_reply_to_user_id', 'favorite_count', 'text', 'place', 'quoted_status_permalink', 'lang', 'quote_count', 'favorited', 'coordinates', 'truncated', 'timestamp_ms', 'reply_count', 'entities', 'quoted_status_id_str', 'contributors', 'user', 'challenge_id', 'label'])

Keys in user dict:
dict_keys(['utc_offset', 'profile_image_url_https', 'listed_count', 'profile_background_image_url', 'default_profile_image', 'favourites_count', 'description', 'created_at', 'is_translator', 'profile_background_image_url_https', 'protected', 'profile_link_color', 'translator_type', 'geo_enabled', 'profile_background_color', 'lang', 'profile_sidebar_border_color', 'profile_text_color', 'profile_image_url', 'time_zone', 'url'

In [ ]:
# Add this debug code to check
with open("data/train.jsonl", "r", encoding="utf-8") as f:
    tweets = [json.loads(line) for line in f]

# Check the challenge_id distribution
challenge_ids = [t.get("challenge_id") for t in tweets]
print(f"Total tweets: {len(tweets)}")
print(f"Unique challenge_ids: {len(set(challenge_ids))}")
print(f"Average tweets per challenge_id: {len(tweets) / len(set(challenge_ids)):.2f}")

# Check if tweets with same challenge_id have same user attributes
from collections import defaultdict
cid_to_users = defaultdict(list)
for t in tweets[:1000]:  # sample first 1000
    cid = t.get("challenge_id")
    user_profile_img = t.get("user", {}).get("profile_image_url_https")
    cid_to_users[cid].append(user_profile_img)

# If challenge_id is per user, all tweets with same challenge_id should have same user attributes
print("\nChecking if same challenge_id = same user:")
for cid, imgs in list(cid_to_users.items())[:5]:
    print(f"  challenge_id {cid}: {len(set(imgs))} unique profile images (should be 1)")

Total tweets: 154914
Unique challenge_ids: 154914
Average tweets per challenge_id: 1.00

Checking if same challenge_id = same user:
  challenge_id 1: 1 unique profile images (should be 1)
  challenge_id 3: 1 unique profile images (should be 1)
  challenge_id 5: 1 unique profile images (should be 1)
  challenge_id 6: 1 unique profile images (should be 1)
  challenge_id 7: 1 unique profile images (should be 1)


: 

2️⃣ Preprocess text

In [14]:
# ------------------------------
# 1. Text preprocessing
# ------------------------------
def preprocess_text(t):
    if not isinstance(t, str):
        return ""
    t = t.strip()
    # Optional: add more preprocessing like lowercasing, URL removal, emoji handling, etc.
    return t

train_df[TEXT_COL] = train_df[TEXT_COL].apply(preprocess_text)
kaggle_df[TEXT_COL] = kaggle_df[TEXT_COL].apply(preprocess_text)

# ------------------------------
# 2. Optional: drop list/dict columns if structured features exist
# ------------------------------
def drop_list_columns(df):
    bad_cols = [c for c in df.columns if df[c].apply(lambda x: isinstance(x, (list, dict))).any()]
    return df.drop(columns=bad_cols)

train_df_clean = drop_list_columns(train_df)
kaggle_df_clean = drop_list_columns(kaggle_df)

# ------------------------------
# 3. Synchronize columns
# ------------------------------
common_cols = list(set(train_df_clean.columns) & set(kaggle_df_clean.columns))
for col in [TEXT_COL, TWEET_ID_COL, USER_ID_COL]:
    if col in common_cols:
        common_cols.remove(col)

train_df_clean = train_df_clean[[TEXT_COL, TWEET_ID_COL, USER_ID_COL] + common_cols]
kaggle_df_clean = kaggle_df_clean[[TEXT_COL, TWEET_ID_COL, USER_ID_COL] + common_cols]

# ------------------------------
# 4. Drop columns with NaN in train or kaggle (except IDs and text)
# ------------------------------
nan_train = train_df_clean.isna().any()
nan_kaggle = kaggle_df_clean.isna().any()
to_drop = nan_train[nan_train].index.union(nan_kaggle[nan_kaggle].index).tolist()
for col in [TEXT_COL, TWEET_ID_COL, USER_ID_COL]:
    if col in to_drop:
        to_drop.remove(col)

train_df_proc = train_df_clean.drop(columns=to_drop)
kaggle_df_proc = kaggle_df_clean.drop(columns=to_drop)

# ------------------------------
# 5. Optional: structured features
# ------------------------------
structured_cols = [c for c in train_df_proc.columns if c not in [TEXT_COL, TWEET_ID_COL, USER_ID_COL]]
numeric_cols = train_df_proc[structured_cols].select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = train_df_proc[structured_cols].select_dtypes(include=["object", "bool"]).columns.tolist()

# ------------------------------
# 6. Save preprocessed DataFrames for Step 3
# ------------------------------
train_df_proc.to_pickle("data/train_df_proc.pkl")
kaggle_df_proc.to_pickle("data/kaggle_df_proc.pkl")

print("Preprocessing complete. Train shape:", train_df_proc.shape)


Preprocessing complete. Train shape: (154914, 3)


3️⃣ Prepare embedding function (CamemBERT)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = CamembertTokenizer.from_pretrained("camembert-base")
model = CamembertModel.from_pretrained("camembert-base")
model.to(device)
model.eval()

def embed_texts(texts, batch_size=16):
    all_embeddings = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size)):
            batch = texts[i:i+batch_size]
            enc = tokenizer(batch, padding=True, truncation=True,
                            max_length=128, return_tensors="pt")
            
            input_ids = enc["input_ids"].to(device)
            attention_mask = enc["attention_mask"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            cls_emb = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            all_embeddings.append(cls_emb)

    return np.vstack(all_embeddings)
    

4️⃣ Embed the tweets in order

In [ ]:
# Extract raw texts
train_texts = train_df_proc["clean_text"].astype(str).tolist()
kaggle_texts = kaggle_df_proc["clean_text"].astype(str).tolist()

# Extract tweet IDs and user IDs for later aggregation
train_tweet_ids = train_df_proc["tweet_id"].tolist() # See data structure to fix
train_user_ids = train_df_proc["user_id"].tolist() # See data structure to fix

kaggle_tweet_ids = kaggle_df_proc["tweet_id"].tolist()
kaggle_user_ids = kaggle_df_proc["user_id"].tolist()

# ------------------------------
# Embed tweets using CamemBERT
# ------------------------------

print("Embedding TRAIN tweets...")
train_embeddings = embed_texts(train_texts, batch_size=16)
print("Done. Train embeddings shape:", train_embeddings.shape)

print("Embedding KAGGLE tweets...")
kaggle_embeddings = embed_texts(kaggle_texts, batch_size=16)
print("Done. Kaggle embeddings shape:", kaggle_embeddings.shape)


Embedding TRAIN tweets...


  0%|          | 0/9683 [00:00<?, ?it/s]

100%|██████████| 9683/9683 [04:25<00:00, 36.53it/s]


Done. Train embeddings shape: (154914, 768)
Embedding KAGGLE tweets...


100%|██████████| 6462/6462 [02:56<00:00, 36.64it/s]


Done. Kaggle embeddings shape: (103380, 768)
Embeddings and IDs saved for hierarchical aggregation.


5️⃣ Save tweet IDs + embeddings + user IDs

In [4]:
EMB_DIR = "./data/embeddings"

In [ ]:
# ------------------------------
# Save embeddings + IDs for hierarchical mapping
# ------------------------------

os.makedirs(EMB_DIR, exist_ok=True)

np.save(os.path.join(EMB_DIR, "train_embeddings.npy"), train_embeddings)
np.save(os.path.join(EMB_DIR, "train_tweet_ids.npy"), np.array(train_tweet_ids))
np.save(os.path.join(EMB_DIR, "train_user_ids.npy"), np.array(train_user_ids))

np.save(os.path.join(EMB_DIR, "kaggle_embeddings.npy"), kaggle_embeddings)
np.save(os.path.join(EMB_DIR, "kaggle_tweet_ids.npy"), np.array(kaggle_tweet_ids))
np.save(os.path.join(EMB_DIR, "kaggle_user_ids.npy"), np.array(kaggle_user_ids))

print("Embeddings and IDs saved for hierarchical aggregation.")

Load embeddings (checkpoint, don't run if running entire notebook)

In [15]:
train_df_proc = pd.read_pickle("data/train_df_proc.pkl")
kaggle_df_proc = pd.read_pickle("data/kaggle_df_proc.pkl")

train_embeddings = np.load(os.path.join(EMB_DIR, "train_embeddings.npy"))
train_tweet_ids = np.load(os.path.join(EMB_DIR, "train_tweet_ids.npy"))
train_user_ids = np.load(os.path.join(EMB_DIR, "train_user_ids.npy"), allow_pickle=True)

kaggle_embeddings = np.load(os.path.join(EMB_DIR, "kaggle_embeddings.npy"))
kaggle_tweet_ids = np.load(os.path.join(EMB_DIR, "kaggle_tweet_ids.npy"))
kaggle_user_ids = np.load(os.path.join(EMB_DIR, "kaggle_user_ids.npy"), allow_pickle=True)

6️⃣ Build user → list of tweet embeddings

In [18]:
print(train_df_proc)

                                               clean_text  \
0                                 C’est exactement ça ...   
1       Depuis un certain temps, Le Gorafi n'a même pl...   
2       @RomainSchulz @OlivierPicot2 @Simonnet2 @saraa...   
3       Renforcer les capacités de dépistages et les a...   
4       @Kidadou @natsu_luffy0832 @olivierveran On me ...   
...                                                   ...   
154909  Baway, le virus vient à l'école juste pour app...   
154910  Incroyable : l’#Etat #belge #CONDAMNÉ à lever ...   
154911    Les collègues qui parlent H24 du Covid rooooooo   
154912  Covid-19 : point de la situation de ce Mercred...   
154913  @f_philippot Ça se voit que vous n'avez perdu ...   

                   tweet_id user_id  
0       1372171356809400322    None  
1       1372171385049604098    None  
2       1372171603874881537    None  
3       1372171610757627904    None  
4       1372171629070016513    None  
...                     ...     ...  
1

In [ ]:
from collections import defaultdict

def build_user_tweet_dict(embeddings, user_ids):
    user_tweets = defaultdict(list)
    for emb, uid in zip(embeddings, user_ids):
        if uid is not None:
            user_tweets[uid].append(emb)
    # Convert each user's list to a numpy array
    for uid in user_tweets:
        user_tweets[uid] = np.vstack(user_tweets[uid])
    return user_tweets

train_user_tweets = build_user_tweet_dict(train_embeddings, train_df_proc["user"])
kaggle_user_tweets = build_user_tweet_dict(kaggle_embeddings, kaggle_df_proc["user_id"])

print(f"Number of users (train): {len(train_user_tweets)}")
print(f"Number of users (kaggle): {len(kaggle_user_tweets)}")

Number of users (train): 0
Number of users (kaggle): 0


7️⃣ (Optional) Save user-level embeddings dictionary

In [ ]:
with open(f"{EMB_DIR}/train_user_tweet_embeddings.pkl", "wb") as f:
    pickle.dump(train_user_tweets, f)

with open(f"{EMB_DIR}/kaggle_user_tweet_embeddings.pkl", "wb") as f:
    pickle.dump(kaggle_user_tweets, f)

print("Saved user-level embeddings dictionaries.")


Saved user-level embeddings dictionaries.


## B. Model

Step 1 — Build a UserDataset

Instead of one row per tweet, each sample is now all tweets for one user:

In [ ]:
class UserDataset(Dataset):
    def __init__(self, user_tweet_embeddings, user_labels, max_tweets=None):
        """
        user_tweet_embeddings: dict[user_id] -> list of tweet embeddings (np.array)
        user_labels: dict[user_id] -> label (0/1)
        max_tweets: maximum number of tweets per user; truncate/pad if needed
        """
        self.user_ids = list(user_tweet_embeddings.keys())
        self.user_tweet_embeddings = user_tweet_embeddings
        self.user_labels = user_labels
        self.max_tweets = max_tweets

    def __len__(self):
        return len(self.user_ids)

    def __getitem__(self, idx):
        uid = self.user_ids[idx]
        tweets = self.user_tweet_embeddings[uid]
        tweets = np.stack(tweets)  # shape: [num_tweets, emb_dim]

        if self.max_tweets is not None:
            # truncate or pad
            if tweets.shape[0] > self.max_tweets:
                tweets = tweets[:self.max_tweets]
            else:
                pad_len = self.max_tweets - tweets.shape[0]
                pad = np.zeros((pad_len, tweets.shape[1]))
                tweets = np.vstack([tweets, pad])

        label = self.user_labels[uid]
        return torch.FloatTensor(tweets), torch.LongTensor([label])

Step 2 — Hierarchical Model Skeleton

In [ ]:
class HierarchicalClassifier(nn.Module):
    def __init__(self, emb_dim, hidden_dim=128, num_classes=2):
        super().__init__()
        # Tweet-level encoder
        self.tweet_encoder = nn.Linear(emb_dim, hidden_dim)
        self.tweet_act = nn.ReLU()
        # User-level aggregator (mean pooling)
        self.user_fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        # x: [batch_size, num_tweets, emb_dim]
        h = self.tweet_act(self.tweet_encoder(x))  # [batch, num_tweets, hidden_dim]
        h_user = h.mean(dim=1)  # simple mean pooling over tweets
        out = self.user_fc(h_user)
        return out

You can later replace mean pooling with attention over tweets.

Step 3 — DataLoader and batching

In [ ]:
user_labels = train_df.groupby("user_id")["label"].first().to_dict()

train_dataset = UserDataset(user_tweet_embeddings=train_user_tweets, user_labels=user_labels, max_tweets=50)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

NameError: name 'user_labels' is not defined

Step 4 — Training loop

In [ ]:
model = HierarchicalClassifier(emb_dim=768, hidden_dim=128)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

model.train()
for epoch in range(10):
    for tweets, labels in train_loader:
        optimizer.zero_grad()
        tweets, labels = tweets.to(device), labels.squeeze().to(device)
        out = model(tweets)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()